# [INFO-H515 - Big Data Scalable Analytics](https://uv.ulb.ac.be/course/view.php?id=85246?username=guest)

## TP 3 - Streaming forecasting (RLS and ML) with Spark Structured Streaming

#### *Gianluca Bontempi, Cédric Simar, and Martin Colot - materials from Jacopo De Stefani and Theo Verhelst*

####  2025

This notebook provides an example for performing machine learning analytics on streaming data produced locally.

The first part contains examples of simple __stateless__ processing scripts.

The second part introduces __stateful__ streaming processing in order to perform more complex operations, like cumulative sum, sequential estimation of mean and variance.

Predictive analytics is also implemented thanks to stateful processing.

Two main __streaming learning__ strategies are proposed:
* an online linear learning where the coefficient estimation is achieved using the __recursive least square (RLS)__ algorithm
* a __mini-batch strategy__ where a learning model is fit to most recent data

A __racing__ strategy is implemented as well where two different models (e.g. two linear models with different forgetting factors or a linear and a nonlinear regressor) are fitted to the same data.

This notebook uses:
* [Spark Structured Streaming](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) for processing the streaming data using DataFrames
* [Spark SQL](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html) for processing the dataFrames


See also :
* [Spark streaming](https://spark.apache.org/docs/latest/streaming-programming-guide.html) for processing the streaming data using RDD (legacy code, not updated since 2016 !)


## General imports

In [ ]:
import time
import re, ast
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn import linear_model
from sklearn.linear_model import LinearRegression

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, rand, udf
from pyspark.sql.types import StringType

## Spark Streaming Session

After creating a spark session, we use `spark.readStream` to create a streaming dataFrame. This data frame loads data automatically from the given source : file, Kafka, socket or rate. The `rate` source works [as follows](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) :

_Generates data at the specified number of rows per second, each output row contains a timestamp and value. Where timestamp is a Timestamp type containing the time of message dispatch, and value is of Long type containing the message count, starting from 0 as the first row. This source is intended for testing and benchmarking._

In [ ]:
# Initialize Spark Session (local mode)
spark = SparkSession.builder \
    .appName("StreamingAnalytics") \
    .master("local[2]") \
    .getOrCreate()

#spark.conf.set("spark.sql.shuffle.partitions", "2") # reduce number of partitions to run on google Colab

In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

After that, we use `writeStream` to output the processed stream in local memory under a given query name

In [ ]:
# Create a query to output the data
query = streaming_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("streaming_data") \
    .start()

time.sleep(2)

We retreive the output from the memory using `spark.sql`

In [ ]:
spark.sql("SELECT * FROM streaming_data ORDER BY value DESC LIMIT 10").toPandas()

,timestamp,value
0,2025-03-25 09:38:25.198,24
1,2025-03-25 09:38:24.998,23
2,2025-03-25 09:38:24.798,22
3,2025-03-25 09:38:24.598,21
4,2025-03-25 09:38:24.398,20
5,2025-03-25 09:38:24.198,19
6,2025-03-25 09:38:23.998,18
7,2025-03-25 09:38:23.798,17
8,2025-03-25 09:38:23.598,16
9,2025-03-25 09:38:23.398,15


Finally, we can stop the query

In [ ]:
query.stop()

and stop the spark session if necessary

In [ ]:
# spark.stop()

## Dataframes transformations

To process the streamed data, we can use most of the common operations on DataFrames. For example, `spark.sql.DataFrame.select` allows to create new streaming dataFrames by selecting columns from another one and applying transformations.

In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

In [ ]:
# Process the DataFrame
processed_df = streaming_df.select(
    col("value").alias("id"),
    (rand()*10).alias("x"),                         # generate random values
    (3 * col("x") + 7 + rand() * 2 - 1).alias("y")  # y = 3x + 7 + noise
)

In [ ]:
# Create a query to output the data
query = processed_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("streaming_data") \
    .start()

time.sleep(2)

In [ ]:
spark.sql("SELECT * FROM streaming_data ORDER BY id DESC LIMIT 10").toPandas()

,id,x,y
0,9,2.342644,14.187253
1,8,6.381064,25.216784
2,7,4.984455,22.116129
3,6,7.329596,28.430538
4,5,8.248948,32.209288
5,4,2.872950,16.366616
6,3,8.260830,32.778398
7,2,9.452181,35.684978
8,1,9.326820,34.982520
9,0,0.456423,7.440881


In [ ]:
query.stop()

### Exercise 1.

The following cell simulates the acquisition of data from multiple persons in 4 countries. Each record contains the country where the person is, the device that the person uses and the screen time of the person for a day (how much time the person spent on the device, in hours).

Use Dataframe opetations to find, for each record, if the screen time is greater than 3 hours. Then, for each pair of country and device, count the number of remaining records. Finally, order the pairs according to this count, in decreasing order.

In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 100) \
    .load()

devices = ["Phone", "Laptop", "Watch"]
country = ["Belgium", "France", "Germany", "Luxembourg"]

# Process the DataFrame
input_df = streaming_df.select(
    col("value").alias("id"),
    (udf(lambda x : devices[int(x * len(devices))], StringType())(rand())).alias("device"), # user defined function (UDF) to map random numbers to strings
    (udf(lambda x : country[int(x * len(country))], StringType())(rand())).alias("country"), # user defined function (UDF) to map random numbers to strings
    (rand()*6).alias("screen_time")
)

**Your solution :**

To update the count at each micro batch without recomuting it from scratch, Spark internally uses hidden states.

## Stateful Batch processing

We can define our own states to compute and update complex aggregations over micro bataches.
In the next example, we use custom states to compute the average of a variable and update it at each micro-match.

It is important to note that the state must be stored in an external database to make it persistent across batches when they are processed on different workers (which does not happen if you run the code on google colab but happens in real infratructures).
Here, we use a simple local sqlite database. For real world applications, refer to solutions like [Redis](https://redis.io/docs/latest/integrate/redis-py/), or [delta lake](https://delta.io/)

Note: This is a simple example of the idea behind statefull processing. However to compute the average of all records, it is sufficient to apply the following operation on DataFrames : `processed_df.agg({"value" : "mean"})`

In [ ]:
conn = sqlite3.connect('/content/state.db')
cursor = conn.cursor()

cursor.execute('''DROP TABLE IF EXISTS state''')

In [ ]:
# Function to update state in SQLite
def update_state(batch_df, batch_id):
    if batch_df.count() > 0:
        # Create a new SQLite connection inside the function
        conn = sqlite3.connect('/content/state.db')
        cursor = conn.cursor()

        # Ensure state table exists
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS state (
                total REAL,
                count INTEGER
            )
        ''')

        # Calculate new batch sum and count
        total_sum = batch_df.agg({"value": "sum"}).collect()[0][0]
        count = batch_df.count()

        # Read previous state from SQLite
        cursor.execute('SELECT total, count FROM state')
        row = cursor.fetchone()
        if row:
            prev_total, prev_count = row
        else:
            prev_total, prev_count = 0, 0  # Initialize if no state exists

        # Compute new state
        new_total = prev_total + total_sum
        new_count = prev_count + count
        new_avg = new_total / new_count if new_count > 0 else 0

        # Store updated state in SQLite
        cursor.execute('DELETE FROM state')  # Remove old state (keeping only one row)
        cursor.execute('INSERT INTO state (total, count) VALUES (?, ?)', (new_total, new_count))
        conn.commit()

        # Close SQLite connection
        cursor.close()
        conn.close()

        display(f"Updated State -> total: {new_total}, count: {new_count}, average: {new_avg}")


In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load()

# Process the DataFrame
processed_df = streaming_df.select(
    col("value").alias("id"),
    (rand()*100).alias("value")  # generate random values
)


# Start the streaming query
query = processed_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .foreachBatch(update_state) \
    .start()


time.sleep(10)

'Updated State -> total: 613.2997955711952, count: 10, average: 61.32997955711952'

'Updated State -> total: 969.1432070767469, count: 20, average: 48.457160353837345'

'Updated State -> total: 1268.0427488058303, count: 30, average: 42.268091626861015'

'Updated State -> total: 1743.2985920535202, count: 40, average: 43.582464801338006'

'Updated State -> total: 2152.70178043934, count: 50, average: 43.0540356087868'

'Updated State -> total: 2566.9834479811016, count: 60, average: 42.78305746635169'

'Updated State -> total: 3060.1855991291545, count: 70, average: 43.71693713041649'

'Updated State -> total: 3511.7794082382125, count: 80, average: 43.897242602977656'

'Updated State -> total: 3965.0588530917316, count: 90, average: 44.056209478797015'

In [ ]:
cursor.execute('SELECT total, count FROM state')
row = cursor.fetchone()
avg = 0
if row:
    total, count = row
    avg = total / count if count > 0 else 0
print(f"total : {total}, count : {count}, average : {avg}")

total : 8958.494994376591, count : 200, average : 44.79247497188296


In [ ]:
query.stop()

### Exercise 2.
Complete the following cells to compute the variance and standard deviation of the incoming values using **custom** stateful processing (don't just use `processed_df.agg({"value" : "var"})`).

Sample variance sequential formulation:  $\sigma^2_{(N)} ={S_{(N)}/N}$ where $$S_{(N)}= S_{(N-1)}+N(N-1) (\mu_{(N)}- \mu_{(N-1)})^2 $$
and $\mu_{(N)}$ is the sample average computed after the $N^{th}$ sample

**Your Solution**

In [ ]:
# ...

In [ ]:
def update_state(batch_df, batch_id):
  ...

In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load()

# Process the DataFrame
processed_df = streaming_df.select(
    col("value").alias("id"),
    (rand()*100).alias("value")  # generate random values
)


# Start the streaming query
query = processed_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .foreachBatch(update_state) \
    .start()

In [ ]:
# ...

In [ ]:
query.stop()

## Recursive Least Squares (RLS)

The following equations define a single step of the Recursive Least Squares (with forgetting factor $\nu$).

\begin{equation*}
\begin{cases}
V_{(t)}&=\frac{1}{\nu} \left(V_{(t-1)}
-\frac{V_{(t-1)} x^T_{t} x_{t} V_{(t-1)}}{1+ x_{t} V_{(t-1)} x^T_{t}} \right)\\[3pt]
\alpha_{(t)}&= V_{(t)} x^T_{t} \\[3pt]
e&= y_{t}- x_{t} \hat{\beta}_{(t-1)}  \\[3pt]
\hat{\beta}_{(t)}&=\hat{\beta}_{(t-1)}+ \alpha_{(t)} e \\[3pt]
\end{cases}
\end{equation*}

where $V$ is the covariance matrix and $\beta$ is the set of parameters of the linear model.
Details in the Streaming Analytics slides.

- $\nu = 1$ : do not forget any sample
- $\nu \rightarrow 0$ : train only on the last samples


### Exercise 3.
Write the implementation of `RLS` in order to perform the update of the state of the RLS model according to the equations given above.

Show the Batch MSE using `display()` and show the fitted coefficients

**Your Solution**

In [ ]:
# ...

In [ ]:
def update_rls_state(batch_df, batch_id):
  ...

In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

# Process the DataFrame
processed_df = streaming_df.select(
    col("value").alias("id"),
    (rand()*10).alias("x"),                         # generate random values
    (a * col("x") + b + rand() * 2 - 1).alias("y")  # y = ax + b + noise
)

# Start the streaming query
query = processed_df.writeStream \
    .format("memory") \
    .foreachBatch(update_rls_state) \
    .start()


time.sleep(10)

In [ ]:
# ...

In [ ]:
query.stop()

## Multiple RLS in parallel

### Exercise 4.
Use multiple queries to 2 concurent RLS in parallel with different forgetting factors

Compute and show the batch MSE and total MSE (sum of batch SE divided by the total number of samples) for each model and show the trained coefficients

**Your Solution**

In [ ]:
# ...

## ML batch predictor (racing of several models)

To conclude this session, we implement a parallel racing of multiple machine learning predictors (here *Random Forests* - `rf`, *Linear Model* - `lin`, *Gradient boosting* - `gb`, respectively)

The aim is to use a predictor trained on the all previous batches to predict the labels of the new batch.

To this end, we store all the previous samples in the state and retrain, from scratch at every batch using `sklearn` models.

As we have 3 parallel workers, we only ask one of them to store the samples in the database. We also store the batch number with the samples to avoid training a model with the testing samples if another worker already sotred them in the database.

### Exercise 5.
Implement this racing of models

**Your Solution**

In [ ]:
# ...

In [ ]:
# ...

In [ ]:
# a*x² + bx + c = y
a = 5
b = 9
c = 2

In [ ]:
# Create a streaming DataFrame
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

# Process the DataFrame
processed_df = streaming_df.select(
    col("value").alias("id"),
    (rand()*10).alias("x1"),
    (rand()*10).alias("x2"),
    (a * col("x1")**2 + b * col("x2") + c + rand() * 2 - 1).alias("y")  # y = a*x² + bx + c = y + noise
)

# Start the streaming query
query1 = ...

query2 = ...

query3 = ...


time.sleep(10)

In [ ]:
query1.stop()
query2.stop()
query3.stop()

## Finally

in the end, stop the spark session

In [ ]:
spark.stop()